# Multi-Objective HPO + Warm Start

This notebook runs a tiny two-objective BayesFlow HPO study and warm-starts a second study from the best trial.

In [ ]:
from pathlib import Path

import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np

In [ ]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

validation_data = hpo.generate_validation_dataset(
    simulator=simulator,
    param_keys=["theta"],
    data_keys=["x"],
    sims_per_condition=16,
)

storage_path = Path("hpo_warmstart_demo.db")
if storage_path.exists():
    storage_path.unlink()
storage = f"sqlite:///{storage_path.as_posix()}"

small_study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["theta"],
    data_keys=["x"],
    validation_data=validation_data,
    n_trials=1,
    epochs=1,
    batches_per_epoch=1,
    storage=storage,
    study_name="hpo_small",
    show_progress_bar=False,
)

print(f"Small study trials: {len(small_study.trials)}")

In [ ]:
large_study = hpo.create_study(
    study_name="hpo_large",
    directions=["minimize", "minimize"],
    storage=storage,
    warm_start_from=small_study,
    warm_start_top_k=1,
)

objective = hpo.GenericObjective(
    hpo.ObjectiveConfig(
        simulator=simulator,
        adapter=adapter,
        search_space=hpo.CompositeSearchSpace(
            inference_space=hpo.CouplingFlowSpace(include_optional=False),
            summary_space=hpo.DeepSetSpace(include_optional=False),
            training_space=hpo.TrainingSpace(include_optional=False),
        ),
        validation_data=validation_data,
        epochs=1,
        batches_per_epoch=1,
    )
)

large_study.optimize(objective, n_trials=1, show_progress_bar=False)

print(f"Large study trials: {len(large_study.trials)}")
print(f"Best values: {large_study.best_trials[0].values}")